<a href="https://colab.research.google.com/github/harneet862/ur2phd-assignment/blob/main/part1_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 0 and Part 1: setup, majority baseline, data exploration

## Setup

In [1]:
!pip install -q transformers datasets accelerate sentencepiece scikit-learn
!git clone https://github.com/ilopezgazpio/SemEval-2027-Task-2-DiCo-NLI.git

Cloning into 'SemEval-2027-Task-2-DiCo-NLI'...
remote: Enumerating objects: 518, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 518 (delta 45), reused 42 (delta 42), pack-reused 471 (from 1)
Receiving objects: 100% (518/518), 29.85 MiB | 25.45 MiB/s, done.
Resolving deltas: 100% (268/268), done.


In [2]:
import pandas as pd, os

TASK_DIR = "/content/SemEval-2027-Task-2-DiCo-NLI"
TRAIN_PATH = f"{TASK_DIR}/final_data/train/dico_nli_train_track1_participant_labeled.csv"
DEV_REF_PATH = f"{TASK_DIR}/final_data/dev/dico_nli_dev_track1_reference.csv"
RESULTS_DIR = f"{TASK_DIR}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

## Part 0: majority baseline
Always predict the most frequent train label, then score it with the official scorer to check the environment works.

In [3]:
train = pd.read_csv(TRAIN_PATH)
dev = pd.read_csv(DEV_REF_PATH)

majority = train["label"].mode()[0]
print("Majority label:", majority)

pd.DataFrame({"instance_id": dev["instance_id"], "label": majority}).to_csv(
    f"{RESULTS_DIR}/majority_dev.csv", index=False)

Majority label: BACKWARD_ENTAILMENT


In [4]:
%cd /content/SemEval-2027-Task-2-DiCo-NLI
!python3 -m evaluation_functions \
  --gold final_data/dev/dico_nli_dev_track1_reference.csv \
  --predictions results/majority_dev.csv \
  --output-dir results/majority_scores

/content/SemEval-2027-Task-2-DiCo-NLI
{
  "classification": {
    "accuracy": 0.2878787878787879,
    "confusion_matrix": {
      "BACKWARD_ENTAILMENT": {
        "BACKWARD_ENTAILMENT": 190,
        "EQUIVALENCE": 0,
        "FORWARD_ENTAILMENT": 0,
        "NEGATIVE_OTHER": 0
      },
      "EQUIVALENCE": {
        "BACKWARD_ENTAILMENT": 174,
        "EQUIVALENCE": 0,
        "FORWARD_ENTAILMENT": 0,
        "NEGATIVE_OTHER": 0
      },
      "FORWARD_ENTAILMENT": {
        "BACKWARD_ENTAILMENT": 190,
        "EQUIVALENCE": 0,
        "FORWARD_ENTAILMENT": 0,
        "NEGATIVE_OTHER": 0
      },
      "NEGATIVE_OTHER": {
        "BACKWARD_ENTAILMENT": 106,
        "EQUIVALENCE": 0,
        "FORWARD_ENTAILMENT": 0,
        "NEGATIVE_OTHER": 0
      }
    },
    "macro_f1": 0.11176470588235295,
    "per_label": {
      "BACKWARD_ENTAILMENT": {
        "f1": 0.4470588235294118,
        "false_negative": 0,
        "false_positive": 470,
        "label": "BACKWARD_ENTAILMENT",
        "pr

## Part 1.1: label distribution in train and dev

In [5]:
print("Train shape:", train.shape)
print(train["label"].value_counts(), "\n")
print("Dev shape:", dev.shape)
print(dev["label"].value_counts())

Train shape: (3042, 7)
label
FORWARD_ENTAILMENT     881
BACKWARD_ENTAILMENT    881
EQUIVALENCE            802
NEGATIVE_OTHER         478
Name: count, dtype: int64 

Dev shape: (660, 8)
label
FORWARD_ENTAILMENT     190
BACKWARD_ENTAILMENT    190
EQUIVALENCE            174
NEGATIVE_OTHER         106
Name: count, dtype: int64


In [6]:
# same counts as percentages, side by side
dist = pd.DataFrame({
    "train_n": train["label"].value_counts(),
    "train_%": (train["label"].value_counts(normalize=True) * 100).round(1),
    "dev_n": dev["label"].value_counts(),
    "dev_%": (dev["label"].value_counts(normalize=True) * 100).round(1),
})
print(dist)

                     train_n  train_%  dev_n  dev_%
label                                              
FORWARD_ENTAILMENT       881     29.0    190   28.8
BACKWARD_ENTAILMENT      881     29.0    190   28.8
EQUIVALENCE              802     26.4    174   26.4
NEGATIVE_OTHER           478     15.7    106   16.1


## Part 1.2: three examples per label
The cell below prints 10 candidates per label from train. I picked three of each by hand and explained them in the markdown cell after it.

In [7]:
pd.set_option("display.max_colwidth", 80)
# a different random_state per label, otherwise FORWARD and BACKWARD show the same pairs reversed
for i, lab in enumerate(["EQUIVALENCE", "FORWARD_ENTAILMENT", "BACKWARD_ENTAILMENT", "NEGATIVE_OTHER"]):
    print("=====", lab)
    print(train[train["label"] == lab][["instance_id", "text1", "text2"]].sample(10, random_state=i).to_string())
    print()

===== EQUIVALENCE
                                instance_id                                         text1                        text2
898    dico_phrasis_0000642__en-en__flipped                            Ukraine protesters         Ukrainian protesters
1143  dico_phrasis_0000841__en-en__original                                to another man          toward a second man
975   dico_phrasis_0000699__en-en__original  concerning negotiations over nuclear matters            for nuclear talks
1312   dico_phrasis_0000995__en-en__flipped                                  to Take Over                      assumes
904    dico_phrasis_0000646__en-en__flipped                           S. Korean president       South Korean President
1330   dico_phrasis_0001008__en-en__flipped                                    among dead                       killed
1543  dico_phrasis_0001206__en-en__original                                       The man               the male adult
1411  dico_phrasis_0001082__en

### Chosen examples

For an ordered pair (text1, text2):
- **FORWARD_ENTAILMENT**: if text1 is true, text2 is also true.
- **BACKWARD_ENTAILMENT**: if text2 is true, text1 is also true.
- **EQUIVALENCE**: both texts have the same meaning (entailment in both directions).
- **NEGATIVE_OTHER**: none of the above; neither text entails the other.

| Label | instance_id | text1 | text2 | Why |
|---|---|---|---|---|
| FORWARD_ENTAILMENT | dico_phrasis_0000021__en-en__flipped | Six killed | One dead | If six people were killed, at least one person is dead. |
| FORWARD_ENTAILMENT | dico_phrasis_0000078__en-en__flipped | in Afghan city | in Afghanistan | Something that happened in an Afghan city happened in Afghanistan. |
| FORWARD_ENTAILMENT | dico_phrasis_0000096__en-en__flipped | A man and a woman | A man | If there is a man and a woman, there is a man. |
| BACKWARD_ENTAILMENT | dico_phrasis_0000120__en-en__original | a boat | A white boat | If there is a white boat, there is a boat. |
| BACKWARD_ENTAILMENT | dico_phrasis_0000165__en-en__original | A black dog | A black pit bull dog | A black pit bull dog is a black dog. |
| BACKWARD_ENTAILMENT | dico_phrasis_0000271__en-en__original | Stocks | US stocks | US stocks are stocks. |
| EQUIVALENCE | dico_phrasis_0001221__en-en__flipped | a black and white photo | a monochrome photograph | A black and white photo is a monochrome photograph. |
| EQUIVALENCE | dico_phrasis_0000788__en-en__flipped | upside-down | inverted | Upside-down and inverted mean the same thing here. |
| EQUIVALENCE | dico_phrasis_0000724__en-en__flipped | a man and a woman | A woman and a man | Changing the order of the man and the woman does not change the meaning. |
| NEGATIVE_OTHER | dico_phrasis_0002843__en-en__original | on Friday | on Thursday | Different days, so neither implies the other. |
| NEGATIVE_OTHER | dico_phrasis_0002676__en-en__original | A young boy | A young girl | A boy and a girl are different people, so neither implies the other. |
| NEGATIVE_OTHER | dico_phrasis_0002104__en-en__original | A blonde dog | A little blonde girl | They describe different things; sharing "blonde" does not create entailment. |


## Part 1.3: how reversed pairs are linked
The reference file has a `reverse_pair_id` column. The participant file does not.

In [8]:
print(dev.columns.tolist())
print(dev[["instance_id", "pair_id", "reverse_pair_id", "text1", "text2", "label"]].head(4).to_string())

['instance_id', 'pair_id', 'text1_lang', 'text2_lang', 'text1', 'text2', 'reverse_pair_id', 'label']
                             instance_id               pair_id                        reverse_pair_id                text1                text2                label
0   dico_phrasis_0000002__en-en__flipped  dico_phrasis_0000002  dico_phrasis_0000002__en-en__original  's war shrine visit      's shrine visit   FORWARD_ENTAILMENT
1  dico_phrasis_0000002__en-en__original  dico_phrasis_0000002   dico_phrasis_0000002__en-en__flipped      's shrine visit  's war shrine visit  BACKWARD_ENTAILMENT
2   dico_phrasis_0000004__en-en__flipped  dico_phrasis_0000004  dico_phrasis_0000004__en-en__original    anti-protest laws     anti-protest law   FORWARD_ENTAILMENT
3  dico_phrasis_0000004__en-en__original  dico_phrasis_0000004   dico_phrasis_0000004__en-en__flipped     anti-protest law    anti-protest laws  BACKWARD_ENTAILMENT


### Observations

In the reference file, each reversible row’s reverse_pair_id contains the instance_id of its reversed counterpart. The link is reciprocal: each row points back to the other. The two rows share a pair_id, while their instance_ids distinguish the original and flipped directions. NEGATIVE_OTHER rows have no reverse-pair link.
